# Phase 2: Data Wrangling & Exploratory Data Analysis

This notebook performs a comprehensive data audit, applies statistical outlier detection (comparing IQR and Z-score), implements a biomedical cleaning strategy, justifies categorical encoding choices, and presents 8 visualizations with clinical interpretations.

## 1. Data Audit

Below is the structured data audit checking each variable in the Cardiovascular Disease Dataset for missing values, duplicates, and range constraints.

| Column Name | Data Type | Expected Range / Values | Missing Values | Duplicate Values | Inconsistencies / Anomalies Identified |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `id` | Integer | Unique identifier | None | None | None |
| `age` | Integer (days) | Positive values (e.g., 10,000 - 25,000 days) | None | None | Stored in days rather than years |
| `gender` | Categorical | 1 (women), 2 (men) | None | None | Coded as 1 and 2 rather than 0 and 1 |
| `height` | Integer (cm) | Physical human heights (e.g., 140 - 200 cm) | None | None | Severe outliers below 100 cm and above 200 cm |
| `weight` | Float (kg) | Physical human weights (e.g., 40 - 150 kg) | None | None | Severe outliers below 30 kg |
| `ap_hi` | Integer | Systolic blood pressure (e.g., 90 - 180 mmHg) | None | None | Negative values and extreme values up to 16,000 mmHg |
| `ap_lo` | Integer | Diastolic blood pressure (e.g., 60 - 110 mmHg) | None | None | Negative values and extreme values up to 11,000 mmHg |
| `cholesterol` | Ordinal | 1 (normal), 2 (above normal), 3 (well above) | None | None | Categorical encoding required |
| `gluc` | Ordinal | 1 (normal), 2 (above normal), 3 (well above) | None | None | Categorical encoding required |
| `smoke` | Binary | 0 (no), 1 (yes) | None | None | None |
| `alco` | Binary | 0 (no), 1 (yes) | None | None | None |
| `active` | Binary | 0 (no), 1 (yes) | None | None | None |
| `cardio` | Binary | 0 (no), 1 (yes) | None | None | Target variable (balanced) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_csv('../data/cardio_train.csv', sep=';')
print("Shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Null counts:\n", df.isnull().sum())
df.head()

## 2. Outlier Detection (IQR vs Z-score)

We implement both the Interquartile Range (IQR) and standard Z-score methods to identify unphysical anomalies in blood pressure columns (`ap_hi` and `ap_lo`).

In [ ]:
q1_hi = df['ap_hi'].quantile(0.25)
q3_hi = df['ap_hi'].quantile(0.75)
iqr_hi = q3_hi - q1_hi
lower_hi = q1_hi - 1.5 * iqr_hi
upper_hi = q3_hi + 1.5 * iqr_hi

q1_lo = df['ap_lo'].quantile(0.25)
q3_lo = df['ap_lo'].quantile(0.75)
iqr_lo = q3_lo - q1_lo
lower_lo = q1_lo - 1.5 * iqr_lo
upper_lo = q3_lo + 1.5 * iqr_lo

outliers_iqr = df[
    (df['ap_hi'] < lower_hi) | (df['ap_hi'] > upper_hi) |
    (df['ap_lo'] < lower_lo) | (df['ap_lo'] > upper_lo)
]
print("IQR Outliers Count:", len(outliers_iqr))

In [ ]:
z_hi = np.abs(stats.zscore(df['ap_hi']))
z_lo = np.abs(stats.zscore(df['ap_lo']))
outliers_z = df[(z_hi > 3) | (z_lo > 3)]
print("Z-score Outliers Count:", len(outliers_z))

In [ ]:
print("IQR Outliers %:", len(outliers_iqr) / len(df) * 100)
print("Z-score Outliers %:", len(outliers_z) / len(df) * 100)

### Outlier Comparison and Clean-up Justification

The Z-score method identifies fewer outliers than the IQR method because extreme biomedical anomalies (such as blood pressure readings of 11,000 or 16,000 mmHg) artificially inflate the mean and standard deviation, pulling the Z-score boundaries outwards and leading to failure in detecting smaller but still unphysical outliers. The IQR method is more robust to these extreme leverage points but still flags physically valid high blood pressure readings (e.g., systolic of 180 mmHg) as outliers.

Therefore, a strict physical biomedical filtering strategy is preferred. We filter blood pressure and body measurements to keep only records corresponding to realistic human physiology: systolic blood pressure between 80 and 220 mmHg, diastolic blood pressure between 50 and 140 mmHg, systolic blood pressure strictly greater than diastolic, height between 130 and 220 cm, and weight between 40 and 180 kg.

In [ ]:
df_clean = df[
    (df['ap_hi'] >= 80) & (df['ap_hi'] <= 220) &
    (df['ap_lo'] >= 50) & (df['ap_lo'] <= 140) &
    (df['ap_hi'] > df['ap_lo']) &
    (df['height'] >= 130) & (df['height'] <= 220) &
    (df['weight'] >= 40) & (df['weight'] <= 180)
].copy()
print("Cleaned dataset shape:", df_clean.shape)
print("Removed records count:", len(df) - len(df_clean))

## 3. Categorical Encoding

For columns `cholesterol` and `gluc` (glucose), which are ordinally coded as 1 (normal), 2 (above normal), and 3 (well above normal), we choose one-hot encoding. This prevents the machine learning models from assuming a strictly linear interval relationship between categories (e.g., that level 3 is exactly three times level 1). Columns `smoke`, `alco`, and `active` are already binarily coded as 0 and 1, so they do not require additional transformations. The `gender` variable is binarily encoded (1 and 2), which we convert to 0 and 1.

In [ ]:
df_clean['gender'] = df_clean['gender'] - 1
df_encoded = pd.get_dummies(df_clean, columns=['cholesterol', 'gluc'], drop_first=True)
df_encoded.head()

## 4. Visualizations

### Chart 1: Distribution of Patient Age

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['age'] / 365.25, bins=30, kde=True, color='teal')
plt.title('Age Distribution (Years)')
plt.xlabel('Age (Years)')
plt.ylabel('Count')
plt.show()

**Interpretation:** The age distribution shows that the majority of patients in the study are middle-aged and older, centered between 40 and 65 years. The curve is slightly left-skewed, indicating a higher representation of older individuals compared to younger demographics. This distribution suggests the model will be well-suited to identify age-related cardiovascular risk factors in this primary patient group.

### Chart 2: Distribution of Systolic and Diastolic Blood Pressure

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(df_clean['ap_hi'], label='Systolic (ap_hi)', fill=True, color='coral')
sns.kdeplot(df_clean['ap_lo'], label='Diastolic (ap_lo)', fill=True, color='royalblue')
plt.title('Blood Pressure Distributions (Cleaned)')
plt.xlabel('Value (mmHg)')
plt.ylabel('Density')
plt.legend()
plt.show()

**Interpretation:** The blood pressure density curves show distinct peaks around standard physiological baselines, with systolic hovering around 120 mmHg and diastolic around 80 mmHg. There is a visible right tail in the systolic distribution, highlighting the presence of hypertensive patients. This variance in blood pressure readings is crucial for training a classifier on cardiac disease patterns.

### Chart 3: Distribution of BMI

In [ ]:
bmi_series = df_clean['weight'] / ((df_clean['height'] / 100) ** 2)
plt.figure(figsize=(8, 5))
sns.histplot(bmi_series, bins=30, kde=True, color='purple')
plt.title('Body Mass Index (BMI) Distribution')
plt.xlabel('BMI (kg/m²)')
plt.ylabel('Count')
plt.show()

**Interpretation:** The computed Body Mass Index (BMI) distribution shows a peak around 26-28 kg/m², which falls into the overweight category. A significant portion of the cohort has a BMI exceeding 30, representing varying classes of obesity. The right-skewed distribution highlights that weight-related metabolic variables are prominent characteristics of the study population.

### Chart 4: Feature Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

**Interpretation:** The correlation heatmap reveals a moderate positive correlation between blood pressure values and the target cardiovascular disease status. The correlation coefficients among physical metrics (height and weight) are positive as expected. Lifestyle habits like smoking and alcohol consumption display weak positive correlations with each other, but generally have low direct linear correlation with the target label.

### Chart 5: Target Variable Balance

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='cardio', data=df_clean, palette='Set2')
plt.title('Target Class Balance (cardio)')
plt.xlabel('Cardiovascular Disease Presence')
plt.ylabel('Count')
plt.show()

**Interpretation:** The target distribution bar plot shows that the dataset is highly balanced, with approximately an equal split between healthy patients (0) and patients diagnosed with cardiovascular disease (1). This balance prevents the model from developing a majority-class bias during training. Consequently, evaluation metrics like classification accuracy will remain reliable benchmarks alongside the F1-score.

### Chart 6: Cardiovascular Disease Rate by Cholesterol Level

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='cholesterol', y='cardio', data=df_clean, palette='viridis')
plt.title('Cardiovascular Disease Rate by Cholesterol Level')
plt.xlabel('Cholesterol Level')
plt.ylabel('CVD Rate')
plt.show()

**Interpretation:** The bar chart shows a clear stepwise increase in the rate of cardiovascular disease as cholesterol levels rise from normal (1) to above normal (2) and well above normal (3). Patients with the highest cholesterol level have a disease rate near 75%, compared to roughly 44% for those with normal cholesterol. This demonstrates that cholesterol levels are strong clinical predictors of cardiovascular risk.

### Chart 7: BMI vs Cardiovascular Disease Status

In [ ]:
df_temp = df_clean.copy()
df_temp['bmi'] = df_temp['weight'] / ((df_temp['height'] / 100) ** 2)
plt.figure(figsize=(8, 5))
sns.boxplot(x='cardio', y='bmi', data=df_temp, palette='Set3')
plt.title('BMI by Cardiovascular Disease Status')
plt.xlabel('Cardiovascular Disease')
plt.ylabel('BMI (kg/m²)')
plt.show()

**Interpretation:** The box plot illustrates that patients diagnosed with cardiovascular disease (cardio = 1) tend to have a higher median BMI compared to their healthy counterparts. The interquartile range for the disease group is also shifted upwards, pointing to a consistent association between elevated body mass and disease presence. This suggests that metabolic burden, as measured by BMI, is a key component of the model's feature space.

### Chart 8: Pulse Pressure vs CVD Status (Age < 55) - The Surprising Insight

In [ ]:
df_temp['pulse_pressure'] = df_temp['ap_hi'] - df_temp['ap_lo']
df_under_55 = df_temp[df_temp['age'] / 365.25 < 55]
plt.figure(figsize=(8, 5))
sns.boxplot(x='cardio', y='pulse_pressure', data=df_under_55, palette='coolwarm')
plt.title('Pulse Pressure by CVD Status (Age < 55)')
plt.xlabel('Cardiovascular Disease')
plt.ylabel('Pulse Pressure (mmHg)')
plt.show()

**Interpretation:** This box plot highlights our surprising insight that the engineered feature, Pulse Pressure (`ap_hi - ap_lo`), shows a very clear separation between healthy and diseased individuals under 55. While raw systolic pressure showed substantial overlap, the pulse pressure median is significantly elevated in the diseased group. This confirms that arterial stiffness, captured by pulse pressure, serves as a superior clinical separator compared to raw systolic pressure alone for younger demographics.